# 02 — SAS_RAG: Single Agent with Retrieval

```
User Question
     │
     ▼
Enterprise Data Analyst (single agent)
     │
     ├─ Step 1: Retrieve semantic context  (Vector Store, top-k=5)
     ├─ Step 2: Create execution plan      (LLM #1)
     ├─ Step 3: Generate SQL query          (LLM #2)
     ├─ Step 4: Execute SQL                 (Spark)
     └─ Step 5: Interpret results            (LLM #3)
     │
     ▼
Final Answer
```

## Properties
- Same single agent as SAS, but with a retrieval step added
- Base context: technical schema only (no full semantic layer)
- Semantic context retrieved per question (top-5 chunks, cosine similarity, threshold 0.3)
- 3 LLM calls + 1 retrieval step per question
- Isolates the effect of retrieval vs direct semantic injection

In [0]:
import time
import json
import re
import uuid
from datetime import datetime
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================
ARCHITECTURE = "SAS_RAG"

# LLM client — shared from mt_config (429-resilient, max_retries=5)
client = LLM_CLIENT

MODEL_ENDPOINT = MT_MODEL_ENDPOINT  # from mt_config
AGENT_PROFILE = ANALYST_PROFILE     # from mt_config

# Build base context (technical schema only — semantic is retrieved)
schema_context_dict = build_unified_context(ARCHITECTURE)
schema_context_str = format_unified_context_for_prompt(schema_context_dict, include_semantic=False)

print(f"✓ SAS_RAG configured")
print(f"  Model:           {MODEL_ENDPOINT}")
print(f"  Architecture:    {ARCHITECTURE}")
print(f"  Semantic:        retrieved (top-{RETRIEVAL_PARAMS['top_k']})")
print(f"  Base context:    {len(schema_context_str):,} chars (technical only)")

In [0]:
# ============================================================
# SAS_RAG WORKFLOW — Single Agent with Retrieval
# ============================================================
# Step 1: RETRIEVE — Get relevant semantic chunks
# Step 2: PLAN    — Create execution plan with retrieved context
# Step 3: SQL     — Generate SQL query
# Step 4: EXECUTE — Run SQL on Spark
# Step 5: INTERPRET — Generate business answer
# ============================================================

def execute_sql_query(sql: str) -> str:
    """Execute SQL and return results as formatted string."""
    try:
        result_df = spark.sql(sql).toPandas()
        n_rows = len(result_df)
        if n_rows > 30:
            return f"({n_rows} rows total, showing first 30)\n" + result_df.head(30).to_string(index=False)
        return result_df.to_string(index=False)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"


def extract_sql_from_response(text: str) -> str:
    """Extract SQL query from LLM response text."""
    match = re.search(r"```(?:sql)?\s*(.+?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    lines = text.strip().split("\n")
    sql_lines = []
    capture = False
    for line in lines:
        stripped = line.strip().upper()
        if stripped.startswith(("SELECT", "WITH")):
            capture = True
        if capture:
            sql_lines.append(line)
    if sql_lines:
        return "\n".join(sql_lines).strip()
    if any(kw in text.upper() for kw in ["SELECT", "FROM"]):
        return text.strip()
    return None


def run_sas_rag(question_dict: dict) -> dict:
    """
    Execute the SAS_RAG workflow for one question.

    Args:
        question_dict: dict with id, question, expected_answer, difficulty, etc.

    Returns:
        dict with answer, plan, sql, metrics, retrieved_chunks, latency, tokens, etc.
    """
    start_time = time.time()
    total_prompt_tokens = 0
    total_completion_tokens = 0
    llm_calls = 0
    steps_log = []

    question = question_dict["question"]
    question_id = question_dict["id"]

    try:
        # ---- STEP 1: RETRIEVE ----
        step_start = time.time()
        retrieved_chunks = retrieve_semantic_context(question)
        retrieved_context = format_retrieved_context_for_prompt(retrieved_chunks)
        step_latency = time.time() - step_start

        steps_log.append({
            "step": 1, "agent": "Enterprise Data Analyst", "action": "retrieve",
            "status": "SUCCESS", "latency_ms": round(step_latency * 1000, 1),
            "details": f"{len(retrieved_chunks)} chunks retrieved"
        })

        # Full context = technical schema + retrieved semantic
        full_context = f"{schema_context_str}\n\n{retrieved_context}"

        # ---- STEP 2: PLAN ----
        step_start = time.time()
        plan_prompt = f"""{AGENT_PROFILE}

AVAILABLE CONTEXT:
{full_context}

QUESTION: {question}

Create a concise execution plan to answer this question using SQL.
List the tables needed, any filters/aggregations, and the approach.
If the question cannot be answered with available data, respond with CANNOT_ANSWER."""

        resp_plan = client.chat.completions.create(
            model=MODEL_ENDPOINT,
            messages=[{"role": "user", "content": plan_prompt}],
            temperature=LLM_PARAMS["plan"]["temperature"],
            max_tokens=LLM_PARAMS["plan"]["max_tokens"],
            **_seed_kwargs()
        )
        plan = resp_plan.choices[0].message.content
        total_prompt_tokens += resp_plan.usage.prompt_tokens
        total_completion_tokens += resp_plan.usage.completion_tokens
        llm_calls += 1
        step_latency = time.time() - step_start

        steps_log.append({
            "step": 2, "agent": "Enterprise Data Analyst", "action": "plan",
            "status": "SUCCESS", "latency_ms": round(step_latency * 1000, 1)
        })

        # Check CANNOT_ANSWER
        if "CANNOT_ANSWER" in plan.upper():
            latency = time.time() - start_time
            return {
                "answer": plan, "plan": plan, "sql_generated": None, "sql_results": None,
                "sql_execution_status": "not_applicable",
                "retrieved_chunks": retrieved_chunks, "steps_log": steps_log,
                "prompt_tokens": total_prompt_tokens, "completion_tokens": total_completion_tokens,
                "total_tokens": total_prompt_tokens + total_completion_tokens,
                "latency_seconds": round(latency, 2), "llm_calls": llm_calls,
                "success": True, "error": None, "is_mock": False
            }

        # ---- STEP 3: SQL GENERATION ----
        step_start = time.time()
        sql_prompt = f"""{AGENT_PROFILE}

AVAILABLE CONTEXT:
{full_context}

QUESTION: {question}
PLAN: {plan}

SQL RULES:
{chr(10).join('- ' + r for r in SQL_SAFETY_RULES)}

Generate a single SQL query to answer this question. Return ONLY the SQL inside ```sql``` blocks."""

        resp_sql = client.chat.completions.create(
            model=MODEL_ENDPOINT,
            messages=[{"role": "user", "content": sql_prompt}],
            temperature=LLM_PARAMS["sql"]["temperature"],
            max_tokens=LLM_PARAMS["sql"]["max_tokens"],
            **_seed_kwargs()
        )
        sql_response = resp_sql.choices[0].message.content
        total_prompt_tokens += resp_sql.usage.prompt_tokens
        total_completion_tokens += resp_sql.usage.completion_tokens
        llm_calls += 1
        sql_query = extract_sql_from_response(sql_response)
        step_latency = time.time() - step_start

        steps_log.append({
            "step": 3, "agent": "Enterprise Data Analyst", "action": "generate_sql",
            "status": "success" if sql_query else "failed", "latency_ms": round(step_latency * 1000, 1)
        })

        # ---- STEP 4: EXECUTE SQL ----
        sql_results = None
        sql_exec_status = "failed"
        if sql_query:
            step_start = time.time()
            sql_results = execute_sql_query(sql_query)
            sql_exec_status = "failed" if sql_results.startswith("SQL ERROR") else "success"
            step_latency = time.time() - step_start

            steps_log.append({
                "step": 4, "agent": "Enterprise Data Analyst", "action": "execute_sql",
                "status": sql_exec_status, "latency_ms": round(step_latency * 1000, 1)
            })

        # ---- STEP 5: INTERPRET ----
        step_start = time.time()
        interpret_prompt = f"""{AGENT_PROFILE}

QUESTION: {question}
PLAN: {plan}
SQL QUERY: {sql_query or 'None generated'}
SQL RESULTS: {sql_results or 'No results'}

Provide a concise business answer based on the SQL results.
Cite specific numbers from the results. Do not invent values not present in the data.
If results are empty or errored, explain honestly what happened."""

        resp_interpret = client.chat.completions.create(
            model=MODEL_ENDPOINT,
            messages=[{"role": "user", "content": interpret_prompt}],
            temperature=LLM_PARAMS["interpret"]["temperature"],
            max_tokens=LLM_PARAMS["interpret"]["max_tokens"],
            **_seed_kwargs()
        )
        answer = resp_interpret.choices[0].message.content
        total_prompt_tokens += resp_interpret.usage.prompt_tokens
        total_completion_tokens += resp_interpret.usage.completion_tokens
        llm_calls += 1
        step_latency = time.time() - step_start

        steps_log.append({
            "step": 5, "agent": "Enterprise Data Analyst", "action": "interpret",
            "status": "SUCCESS", "latency_ms": round(step_latency * 1000, 1)
        })

        latency = time.time() - start_time
        return {
            "answer": answer, "plan": plan, "sql_generated": sql_query,
            "sql_results": str(sql_results)[:500] if sql_results else None, "sql_execution_status": sql_exec_status,
            "retrieved_chunks": retrieved_chunks, "steps_log": steps_log,
            "prompt_tokens": total_prompt_tokens, "completion_tokens": total_completion_tokens,
            "total_tokens": total_prompt_tokens + total_completion_tokens,
            "latency_seconds": round(latency, 2), "llm_calls": llm_calls,
            "success": True, "error": None, "is_mock": False
        }

    except Exception as e:
        latency = time.time() - start_time
        return {
            "answer": None, "plan": None, "sql_generated": None, "sql_results": None,
            "sql_execution_status": "failed",
            "retrieved_chunks": retrieved_chunks if 'retrieved_chunks' in dir() else [],
            "steps_log": steps_log,
            "prompt_tokens": total_prompt_tokens, "completion_tokens": total_completion_tokens,
            "total_tokens": total_prompt_tokens + total_completion_tokens,
            "latency_seconds": round(latency, 2), "llm_calls": llm_calls,
            "success": False, "error": str(e), "is_mock": False
        }


print("✓ run_sas_rag() defined")
print("  Workflow: Retrieve → Plan (LLM#1) → SQL (LLM#2) → Execute → Interpret (LLM#3)")
print(f"  LLM calls per question: 3 + retrieval")

In [0]:
# ============================================================
# RUN SAS_RAG ARCHITECTURE
# ============================================================

# Respect outer scope (orchestrator sets this); default to all if standalone
if 'QUESTION_FILTER' not in dir():
    QUESTION_FILTER = None
run_questions = [q for q in EVALUATION_QUESTIONS if QUESTION_FILTER is None or q["id"] in QUESTION_FILTER]
_verbose = 'ARCHITECTURES_TO_RUN' not in dir()  # compact when called from orchestrator

print(f"SAS_RAG: {len(run_questions)} questions | top-{RETRIEVAL_PARAMS['top_k']} retrieval | {MODEL_ENDPOINT}")

all_results = []

for q in run_questions:
    run_id = str(uuid.uuid4())

    try:
        # Execute
        result = run_sas_rag(q)

        # Compute evaluation metrics
        sql_exec_status = result.get("sql_execution_status", "failed")
        metrics = compute_all_metrics(
            run={"answer": result["answer"], "sql_generated": result.get("sql_generated"),
                 "sql_results": result.get("sql_results"), "success": result["success"],
                 "error": result.get("error")},
            question={"expected_answer": q["expected_answer"], "tables_needed": q.get("tables_needed", []),
                      "answerability_label": q.get("answerability_label", "answerable"),
                      "expected_claims": q.get("expected_claims", []),
                      "question": q["question"]},
            mode="SAS_RAG"
        )

        # Compact per-question output
        print(f"  {q['id']}: {result['latency_seconds']:.1f}s | {result['total_tokens']} tok | SQL:{sql_exec_status} | chunks:{len(result.get('retrieved_chunks',[]))}")

        # Store
        all_results.append({
            "run_id": run_id, "question_id": q["id"], "question_text": q["question"],
            "question_type": q["question_type"], "difficulty": q["difficulty"],
            "expected_answer": q["expected_answer"],
            "architecture": ARCHITECTURE, "context_mode": CONTEXT_MODE,
            "semantic_delivery": "retrieved", "profile_name": "Enterprise Data Analyst",
            "model": MODEL_ENDPOINT, "plan": result.get("plan"),
            "sql_generated": result.get("sql_generated"), "sql_results": str(result.get("sql_results", ""))[:500],
            "sql_execution_status": sql_exec_status,
            "generated_answer": result.get("answer"),
            "retrieved_chunks": get_retrieval_log(result.get('retrieved_chunks', [])),
            "latency_seconds": result["latency_seconds"],
            "prompt_tokens": result.get("prompt_tokens", 0),
            "completion_tokens": result.get("completion_tokens", 0),
            "total_tokens": result["total_tokens"],
            "number_of_model_calls": result["llm_calls"],
            "llm_calls": result["llm_calls"],
            # Thesis KPIs (from question metadata)
            "answerable": q.get("answerability_label", "answerable") in ("answerable", "yes", ""),
            "sql_required": bool(q.get("tables_needed")),
            "sql_retry_count": result.get("sql_attempts", 1) - 1,
            # Workflow
            "success": result["success"], "error": result.get("error"),
            **metrics,
        })

    except Exception as _loop_err:
        print(f"  {q['id']}: ✗ FAILED — {type(_loop_err).__name__}: {_loop_err}")
        all_results.append({
            "run_id": run_id, "question_id": q["id"], "question_text": q["question"],
            "question_type": q["question_type"], "difficulty": q["difficulty"],
            "expected_answer": q["expected_answer"],
            "architecture": ARCHITECTURE, "context_mode": CONTEXT_MODE,
            "semantic_delivery": "retrieved", "profile_name": "Enterprise Data Analyst",
            "model": MODEL_ENDPOINT, "plan": None,
            "sql_generated": None, "sql_results": None,
            "sql_execution_status": "failed",
            "generated_answer": None,  # No answer produced — error in "error" field
            "retrieved_chunks": None,
            "latency_seconds": 0.0,
            "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0,
            "number_of_model_calls": 0, "llm_calls": 0,
            "answerable": q.get("answerability_label", "answerable") in ("answerable", "yes", ""),
            "sql_required": bool(q.get("tables_needed")),
            "sql_retry_count": 0,
            "success": False, "error": f"{type(_loop_err).__name__}: {_loop_err}",
        })

results_df = pd.DataFrame(all_results)

print(f"\n{'='*70}")
print(f"SAS_RAG RUN COMPLETE: {len(results_df)} runs")
print(f"{'='*70}")
print(f"  SQL success:     {(results_df['sql_execution_status']=='success').sum()}/{len(results_df)}")
print(f"  Avg latency:     {results_df['latency_seconds'].mean():.2f}s")
print(f"  Total tokens:    {results_df['total_tokens'].sum():,}")
print(f"  Avg correctness: {results_df['answer_correctness_proxy'].mean():.2f}")
print(f"  Avg groundedness:{results_df['groundedness_score'].mean():.2f}")

In [0]:
# ============================================================
# CAPTURE SAS_RAG RESULTS + PERSIST TO DELTA
# ============================================================
sas_rag_results_df = None

if "SAS_RAG" in ARCHITECTURES_TO_RUN and 'results_df' in dir() and len(results_df) > 0:
    sas_rag_results_df = results_df.copy()
    all_experiment_results.extend(results_df.to_dict('records'))
    print(f"✓ SAS_RAG captured: {len(sas_rag_results_df)} runs")
else:
    print("⚠ SAS_RAG not run or no results")

_print_cumulative_comparison()

# --- Persist to Delta ---
try:
    if sas_rag_results_df is not None:
        _sdf = spark.createDataFrame(sas_rag_results_df.astype(str))
        _sdf.write.mode("append").option("mergeSchema", "true").saveAsTable(_RESULTS_TABLE)
        print(f"✓ SAS_RAG appended to {_RESULTS_TABLE}")
except Exception as _e:
    print(f"⚠ Delta persist failed: {_e}")

import gc; gc.collect()